### 1. Importação e localização da base

Importamos a biblioteca Pandas e definimos o caminho do arquivo Excel utilizado no projeto, verificando se a base está disponível.

In [ ]:
from pathlib import Path
import pandas as pd

# Localiza a raiz do projeto
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Caminho esperado da base
ARQUIVO_BASE = (
    ROOT
    / "data"
    / "raw"
    / "BD_Atlas_1991_2025_v1.1_2026.08.06_Consolidado.xlsx"
)

print("Raiz do projeto:", ROOT)
print("Arquivo encontrado:", ARQUIVO_BASE.exists())

### 2. Identificação das planilhas

Localizamos o arquivo Excel e verificamos suas planilhas para selecionar a base corrigida que será utilizada na análise.


In [ ]:
if not ARQUIVO_BASE.exists():
    arquivos_excel = list((ROOT / "data").rglob("*.xlsx"))

    print("Arquivos Excel encontrados:")
    for arquivo in arquivos_excel:
        print(arquivo)

else:
    arquivos_excel = [ARQUIVO_BASE]

if arquivos_excel:
    ARQUIVO_BASE = arquivos_excel[0]

    excel = pd.ExcelFile(ARQUIVO_BASE)

    print("\nArquivo utilizado:", ARQUIVO_BASE)
    print("Planilhas disponíveis:", excel.sheet_names)
else:
    print("Nenhum arquivo Excel foi encontrado dentro de data.")

### 3. Inspeção inicial dos dados

Visualizamos as colunas e as primeiras linhas da planilha para compreender sua estrutura e identificar os principais atributos disponíveis.

In [ ]:
if ARQUIVO_BASE.exists():
    df = pd.read_excel(
        ARQUIVO_BASE,
        sheet_name="Atlas Valores Corrigidos",
        nrows=10
    )

    print("Dimensões da amostra:", df.shape)

    print("\nColunas:")
    print(df.columns.tolist())

    display(df.head())

----------------------------------------------|----------------------------------------

### 4. Carregamento completo da base

Carregamos todos os registros da planilha selecionada e verificamos as dimensões da base para confirmar a quantidade de linhas e colunas.

In [ ]:
df = pd.read_excel(
    ARQUIVO_BASE,
    sheet_name="Atlas Valores Corrigidos"
)

print(f"Quantidade de registros: {len(df):,}")
print(f"Quantidade de colunas: {len(df.columns)}")

display(df.head())

### 5. Valores ausentes e duplicidades

Investigamos a quantidade de valores ausentes, possíveis registros totalmente duplicados e os tipos de dados presentes, identificando pontos de atenção para o pré-processamento.

In [ ]:
# Quantidade de valores ausentes por coluna
ausentes = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

resumo_ausentes = pd.DataFrame({
    "valores_ausentes": ausentes,
    "percentual_ausente": (ausentes / len(df) * 100).round(2)
})

display(resumo_ausentes.head(20))

# Registros inteiramente duplicados
print(
    "Linhas totalmente duplicadas:",
    df.duplicated().sum()
)

# Tipos de dados
display(
    df.dtypes.astype(str)
    .value_counts()
    .rename_axis("tipo")
    .reset_index(name="quantidade_colunas")
)

### 6. Distribuição dos tipos de desastre

Analisamos a frequência das categorias de desastre e seus grupos para compreender a composição da base e identificar os registros relevantes para a classificação.

In [ ]:
print("Categorias de descricao_tipologia:")

display(
    df["descricao_tipologia"]
    .value_counts(dropna=False)
    .rename_axis("descricao_tipologia")
    .reset_index(name="quantidade")
)

print("Categorias de grupo_de_desastre:")

display(
    df["grupo_de_desastre"]
    .value_counts(dropna=False)
    .rename_axis("grupo_de_desastre")
    .reset_index(name="quantidade")
)

### 7. Identificação preliminar das classes

Realizamos uma contagem preliminar dos registros associados a FALTA e EXCESSO, identificando também os registros fora do escopo da classificação binária.

In [ ]:
descricao = (
    df["descricao_tipologia"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Máscaras preliminares: servem para inspecionar os candidatos.
mascara_falta = descricao.str.contains(
    r"estiagem|seca",
    regex=True
)

mascara_excesso = descricao.str.contains(
    r"inunda|enxurrada|alagamento",
    regex=True
)

print("Candidatos a FALTA:", mascara_falta.sum())
print("Candidatos a EXCESSO:", mascara_excesso.sum())
print(
    "Registros que coincidem com ambas as máscaras:",
    (mascara_falta & mascara_excesso).sum()
)
print(
    "Registros que não coincidem com nenhuma:",
    (~mascara_falta & ~mascara_excesso).sum()
)

### 8. Mapeamento do alvo provisório

Mapeamos as tipologias selecionadas para as classes FALTA e EXCESSO, calculando suas frequências e proporções e separando os registros fora do escopo do modelo.

In [ ]:
# Mapeamento explícito das classes
mapa_classes = {
    "Estiagem e Seca": "FALTA",
    "Enxurradas": "EXCESSO",
    "Inundações": "EXCESSO",
    "Alagamentos": "EXCESSO",
}

# Criar alvo provisório: registros fora do escopo ficam como NaN
df["alvo"] = df["descricao_tipologia"].map(mapa_classes)

# Quantidade por classe, incluindo os registros fora do escopo
print("Distribuição do alvo provisório:")

display(
    df["alvo"]
    .value_counts(dropna=False)
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

# Distribuição apenas entre as duas classes válidas
df_modelagem = df[df["alvo"].notna()].copy()

distribuicao = (
    df_modelagem["alvo"]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

distribuicao["percentual"] = (
    distribuicao["quantidade"] / len(df_modelagem) * 100
).round(2)

display(distribuicao)

print("Total de registros para classificação binária:", len(df_modelagem))
print("Registros fora do escopo:", df["alvo"].isna().sum())